In [1]:
import pandas as pd
import numpy as np
import json
import requests
from datetime import datetime
import time
import psycopg2
import psycopg2.extras

In [2]:
API_URL = 'https://www.scstrade.com/MarketStatistics/MS_HistoricalPrices.aspx/chart'

PAGE_URL = 'https://www.scstrade.com/MarketStatistics/MS_HistoricalPrices.aspx'
REQUEST_TIMEOUT = 45
MAX_RETRIES = 3
SLEEP_BETWEEN = 0.4
START_DATE = '05/01/2026'
END_DATE = datetime.now().strftime('%m/%d/%Y')
symbols = [
    'ATRL',
    'BAHL',
    'BOP',
    'DGKC',
    'EFERT',
    'ENGROH',
    'FCCL',
    'FFC',
    'GAL',
    'GHNI',
    'HBL',
    'HUBC',
    'LUCK',
    'MARI',
    'MCB',
    'MEBL',
    'MLCF',
    'NBP',
    'OGDC',
    'PAEL',
    'POL',
    'PPL',
    'PRL',
    'PSO',
    'SAZEW',
    'SEARL',
    'SNGP',
    'SSGC',
    'SYS',
    'UBL'
]

# Optional explicit mapping for symbols that need full label in `par`.
PAR_OVERRIDES = {
    'MEBL': 'MEBL - Meezan Bank Ltd.',
}

def parse_dotnet_ms(value):

    if value is None:

        return pd.NaT

    if isinstance(value, (int, float, np.integer, np.floating)):

        return pd.to_datetime(int(value), unit='ms', utc=True).tz_convert('Asia/Karachi').tz_localize(None)

    s = str(value)

    if '/Date(' in s:

        ms = ''.join(ch for ch in s if ch.isdigit())

        return pd.to_datetime(int(ms), unit='ms', utc=True).tz_convert('Asia/Karachi').tz_localize(None)

    return pd.to_datetime(s, errors='coerce', dayfirst=True)



def extract_records(payload):

    if isinstance(payload, list):

        return payload

    if isinstance(payload, dict):

        if isinstance(payload.get('rows'), list):

            return payload['rows']

        for key in ('d', 'data', 'Data', 'records', 'Records'):

            if key in payload and isinstance(payload[key], list):

                return payload[key]

        if 'd' in payload and isinstance(payload['d'], str):

            try:

                parsed = json.loads(payload['d'])

                if isinstance(parsed, list):

                    return parsed

                if isinstance(parsed, dict):

                    if isinstance(parsed.get('rows'), list):

                        return parsed['rows']

                    for key in ('data', 'Data', 'records', 'Records'):

                        if key in parsed and isinstance(parsed[key], list):

                            return parsed[key]

            except json.JSONDecodeError:

                return []

    return []



def normalize_record(rec, symbol):

    if isinstance(rec, dict) and isinstance(rec.get('cell'), list):

        cell = rec['cell']

        if len(cell) >= 6:

            return {

                'symbol': symbol,

                'date': parse_dotnet_ms(cell[0]),

                'open': pd.to_numeric(cell[1], errors='coerce'),

                'high': pd.to_numeric(cell[2], errors='coerce'),

                'low': pd.to_numeric(cell[3], errors='coerce'),

                'close': pd.to_numeric(cell[4], errors='coerce'),

                'volume': pd.to_numeric(cell[5], errors='coerce'),

            }

    date_value = rec.get('Date') or rec.get('date') or rec.get('TradingDate') or rec.get('x') or rec.get('trading_Date')

    return {

        'symbol': symbol,

        'date': parse_dotnet_ms(date_value),

        'open': pd.to_numeric(rec.get('Open') or rec.get('open') or rec.get('o') or rec.get('trading_open'), errors='coerce'),

        'high': pd.to_numeric(rec.get('High') or rec.get('high') or rec.get('h') or rec.get('trading_high'), errors='coerce'),

        'low': pd.to_numeric(rec.get('Low') or rec.get('low') or rec.get('l') or rec.get('trading_low'), errors='coerce'),

        'close': pd.to_numeric(rec.get('Close') or rec.get('close') or rec.get('c') or rec.get('trading_close'), errors='coerce'),

        'volume': pd.to_numeric(rec.get('Volume') or rec.get('volume') or rec.get('v') or rec.get('trading_vol'), errors='coerce'),

    }



def build_payload(par_value):

    return {

        'par': par_value,

        'date1': START_DATE,

        'date2': END_DATE,

        '_search': False,

        'nd': int(time.time() * 1000),

        'rows': 20,

        'page': 1,

        'sidx': 'trading_Date',

        'sord': 'desc',

    }



def make_session():

    sess = requests.Session()

    # Warm ASP.NET session cookie from page load.

    sess.get(PAGE_URL, headers={'User-Agent': 'Mozilla/5.0'}, timeout=REQUEST_TIMEOUT)

    if 'displayoptions' not in sess.cookies:

        sess.cookies.set('displayoptions', '1', domain='www.scstrade.com', path='/')

    return sess



def fetch_symbol_history(session, symbol):

    par_value = PAR_OVERRIDES.get(symbol, symbol)

    headers = {

        'User-Agent': 'Mozilla/5.0 (X11; Ubuntu; Linux x86_64; rv:148.0) Gecko/20100101 Firefox/148.0',

        'Accept': 'application/json, text/javascript, */*; q=0.01',

        'Accept-Language': 'en-US,en;q=0.9',

        'Content-Type': 'application/json',

        'X-Requested-With': 'XMLHttpRequest',

        'Origin': 'https://www.scstrade.com',

        'Referer': PAGE_URL,

        'DNT': '1',

    }

    for attempt in range(1, MAX_RETRIES + 1):

        try:

            payload = build_payload(par_value)

            resp = session.post(API_URL, data=json.dumps(payload), headers=headers, timeout=REQUEST_TIMEOUT)

            resp.raise_for_status()

            raw = resp.json()

            if isinstance(raw, dict) and raw.get('Message'):

                raise ValueError(raw.get('Message'))

            records = extract_records(raw)

            rows = [normalize_record(r, symbol) for r in records]

            return pd.DataFrame(rows)

        except Exception as exc:

            if attempt == MAX_RETRIES:

                print(f'[ERROR] {symbol} ({par_value}) failed after {MAX_RETRIES} tries: {exc}')

                return pd.DataFrame(columns=['symbol', 'date', 'open', 'high', 'low', 'close', 'volume'])

            time.sleep(1.2 * attempt)



# Quick one-shot probe with the provided sample `par`.

probe_session = make_session()

probe_payload = {

    'par': 'MEBL - Meezan Bank Ltd.',

    'date1': START_DATE,

    'date2': END_DATE,

    '_search': False,

    'nd': int(time.time() * 1000),

    'rows': 20,

    'page': 1,

    'sidx': 'trading_Date',

    'sord': 'desc',

}

probe_resp = probe_session.post(

    API_URL,

    data=json.dumps(probe_payload),

    headers={

        'User-Agent': 'Mozilla/5.0 (X11; Ubuntu; Linux x86_64; rv:148.0) Gecko/20100101 Firefox/148.0',

        'Accept': 'application/json, text/javascript, */*; q=0.01',

        'Content-Type': 'application/json',

        'X-Requested-With': 'XMLHttpRequest',

        'Origin': 'https://www.scstrade.com',

        'Referer': PAGE_URL,

    },

    timeout=REQUEST_TIMEOUT,

)

print('Probe status:', probe_resp.status_code)

print('Probe content-type:', probe_resp.headers.get('content-type'))



session = make_session()

all_frames = []

for i, sym in enumerate(symbols, 1):

    print(f'[{i:02d}/{len(symbols)}] Fetching {sym}...')

    df_sym = fetch_symbol_history(session, sym)

    all_frames.append(df_sym)

    time.sleep(SLEEP_BETWEEN)



raw_df = pd.concat(all_frames, ignore_index=True)

print('\nRaw merged shape:', raw_df.shape)

Probe status: 200
Probe content-type: application/json; charset=utf-8
[01/30] Fetching ATRL...
[02/30] Fetching BAHL...
[03/30] Fetching BOP...
[04/30] Fetching DGKC...
[05/30] Fetching EFERT...
[06/30] Fetching ENGROH...
[07/30] Fetching FCCL...
[08/30] Fetching FFC...
[09/30] Fetching GAL...
[10/30] Fetching GHNI...
[11/30] Fetching HBL...
[12/30] Fetching HUBC...
[13/30] Fetching LUCK...
[14/30] Fetching MARI...
[15/30] Fetching MCB...
[16/30] Fetching MEBL...
[17/30] Fetching MLCF...
[18/30] Fetching NBP...
[19/30] Fetching OGDC...
[20/30] Fetching PAEL...
[21/30] Fetching POL...
[22/30] Fetching PPL...
[23/30] Fetching PRL...
[24/30] Fetching PSO...
[25/30] Fetching SAZEW...
[26/30] Fetching SEARL...
[27/30] Fetching SNGP...
[28/30] Fetching SSGC...
[29/30] Fetching SYS...
[30/30] Fetching UBL...

Raw merged shape: (150, 7)


In [4]:
required_cols = ['symbol', 'date', 'open', 'high', 'low', 'close', 'volume']

for col in required_cols:

    if col not in raw_df.columns:

        raw_df[col] = np.nan



df = raw_df[required_cols].copy()

df['symbol'] = df['symbol'].astype(str).str.upper().str.strip()

df['date'] = pd.to_datetime(df['date'], errors='coerce')

for col in ['open', 'high', 'low', 'close', 'volume']:

    df[col] = pd.to_numeric(df[col], errors='coerce')



df = df.dropna(subset=['symbol', 'date']).drop_duplicates(subset=['symbol', 'date'])

df = df.sort_values(['date', 'symbol']).reset_index(drop=True)



# Remove dates where any OHLCV field is NaN for any stock.

nan_by_date = df.groupby('date')[['open', 'high', 'low', 'close', 'volume']].apply(lambda x: x.isna().any().any())

bad_dates = nan_by_date[nan_by_date].index

df_clean = df[~df['date'].isin(bad_dates)].copy()



print('Dates removed due to NaN OHLCV:', len(bad_dates))

print('Clean shape:', df_clean.shape)

print('Symbols present:', df_clean['symbol'].nunique())

print('Date range:', df_clean['date'].min(), 'to', df_clean['date'].max())
df_clean.head()

Dates removed due to NaN OHLCV: 0
Clean shape: (150, 7)
Symbols present: 30
Date range: 2026-05-04 00:00:00 to 2026-05-08 00:00:00


,symbol,date,open,high,low,close,volume
0,ATRL,2026-05-04,934.00,934.00,891.00,898.24,702816
1,BAHL,2026-05-04,166.80,173.50,166.00,167.63,98041
2,BOP,2026-05-04,34.90,35.70,34.00,34.49,39635378
3,DGKC,2026-05-04,180.01,184.89,172.00,174.85,7413294
4,EFERT,2026-05-04,198.00,200.99,197.01,197.75,677969


In [ ]:
df_pg = df_clean.copy()
df_pg = df_pg[['date', 'symbol', 'open', 'high', 'low', 'close', 'volume']]

# PostgreSQL expects 'NULL' instead of Pandas' 'NaN' for missing values
df_pg = df_pg.replace({np.nan: None})

# Convert the DataFrame into a list of tuples (stripping out the Pandas index automatically)
data_tuples = [tuple(row) for row in df_pg.to_numpy()]

# 2. Connect to the Database
# Get this from Supabase Dashboard -> Settings -> Database -> Connection String (URI)
DATABASE_URL=""

conn = psycopg2.connect(DATABASE_URL)
cursor = conn.cursor()

# 3. The SQL Query
# ON CONFLICT DO NOTHING ensures the script doesn't crash if a row already exists
insert_query = """
    INSERT INTO stock_ohlcv (date, symbol, open, high, low, close, volume)
    VALUES %s
    ON CONFLICT (date, symbol) DO NOTHING;
"""

# 4. Execute the Bulk Insert
try:
    print(f"Starting insertion of {len(df_pg)} rows...")
    
    # execute_values with page_size=10000 sends the data in bulk chunks
    psycopg2.extras.execute_values(
        cursor, 
        insert_query, 
        data_tuples,
        page_size=10000 
    )
    
    # Save the changes
    conn.commit()
    print("Data successfully inserted!")
    
except Exception as e:
    print(f"Error during insertion: {e}")
    # Revert the transaction if something fails
    conn.rollback()
    
finally:
    # Always close your connections!
    cursor.close()
    conn.close()

Starting insertion of 150 rows...
Data successfully inserted!


In [6]:
upsert_query = """
    INSERT INTO stock_predicted_close (date, symbol, close)
    SELECT date, symbol, close
    FROM stock_ohlcv
    ON CONFLICT (date, symbol) 
    DO UPDATE SET close = EXCLUDED.close;
"""

try:
    print("Syncing predicted_close table...")
    conn = psycopg2.connect(DATABASE_URL)
    cursor = conn.cursor()
    
    # Execute the massive SQL upsert command
    cursor.execute(upsert_query)
    conn.commit()
    
    # cursor.rowcount tells you how many rows were inserted or updated
    print(f"✅ Sync complete! {cursor.rowcount} rows processed.")

except Exception as e:
    print(f"❌ Error during sync: {e}")
    if conn:
        conn.rollback()
finally:
    if cursor:
        cursor.close()
    if conn:
        conn.close()

Syncing predicted_close table...
✅ Sync complete! 144072 rows processed.
